# UAE Mobile Intelligence - Master Zone-Quarter Table

Joins the three zone-level tables built so far into one analysis-ready table: one row per
`(H3 res-7 cell, quarter)`, everything downstream (Experience Index, Confidence Score, peer
grouping, anomaly detection, priority) reads from this single file instead of re-joining three
parquet files every time.

- **`ookla_zones_uae.parquet`** -- the measured universe: one row per `(h3_cell, quarter)` with
  test-weighted download/upload/latency, tests, devices. This defines which zone-quarters exist
  in the output (only zones Ookla actually measured get scored -- unmeasured zones belong to a
  separate "eligible universe" table for the T0 coverage audit, not this one).
- **`population_zones_uae.parquet`** -- static (not quarterly): population per h3_cell, summed
  from WorldPop with the national total conserved to the person.
- **`osm_density_zones_uae.parquet`** -- static: building/POI/road density per h3_cell, from the
  Geofabrik OSM extract. Only includes zones with at least one OSM feature.

Population and OSM density are broadcast onto every quarter of a zone via a left join on
`h3_cell` -- they don't change quarter to quarter, only the Ookla measurements do.

In [1]:
from pathlib import Path

import h3
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## Load the three tables

In [2]:
ookla = pd.read_parquet("../data/processed/ookla_zones_uae.parquet")
population = pd.read_parquet("../data/processed/population_zones_uae.parquet")
osm = pd.read_parquet("../data/processed/osm_density_zones_uae.parquet")

print("Ookla zone-quarters:", len(ookla), "| unique zones:", ookla["h3_cell"].nunique())
print("Population zones:   ", len(population))
print("OSM density zones:  ", len(osm))

Ookla zone-quarters: 13597 | unique zones: 3400
Population zones:    7880
OSM density zones:   6575


## Left-join population and OSM density onto the measured (Ookla) universe

Every zone Ookla measured gets a population and density figure, even if that exact H3 cell
never shows up in the population or OSM tables -- those tables only list zones with a nonzero
value (WorldPop's constrained product only records populated pixels; the OSM table only
includes zones with at least one building/road/POI). A zone measured by Ookla but absent from
both is a real zone with zero estimated population and zero OSM features (e.g. open desert or
water's edge that a tile happens to touch) -- not a missing value, so it's filled with 0, not
dropped or left NaN.

`zone_area_km2` is the one field that can't be filled with 0 for a missing row -- every H3 cell
has a real geodesic area regardless of whether OSM has any features in it -- so it's computed
directly from `h3.cell_area` for every zone rather than inherited from the OSM join.

In [3]:
zone_quarter = ookla.merge(population, on="h3_cell", how="left")
zone_quarter = zone_quarter.merge(osm, on="h3_cell", how="left")

assert len(zone_quarter) == len(ookla), "Join must not add or drop zone-quarter rows"
assert zone_quarter["tests"].sum() == ookla["tests"].sum(), "Join must not touch Ookla measurements"

density_cols = [
    "building_count", "building_area_m2", "road_count", "road_length_m", "poi_count",
    "building_footprint_pct", "building_count_per_km2", "poi_count_per_km2", "road_density_km_per_km2",
]
zone_quarter["population"] = zone_quarter["population"].fillna(0.0)
zone_quarter[density_cols] = zone_quarter[density_cols].fillna(0.0)

# zone_area_km2 is a geometric constant, not an OSM-derived count -- recompute for every zone
# (not just the ones missing from the OSM join) so it's never inherited from a stale join.
zone_quarter["zone_area_km2"] = [h3.cell_area(c, unit="km^2") for c in zone_quarter["h3_cell"]]

print("Zone-quarter rows:", len(zone_quarter))
zone_quarter.head()

Zone-quarter rows: 13597


,h3_cell,quarter,n_tiles,tests,devices,tests_loaded_lat,download_mbps,upload_mbps,latency_ms,latency_loaded_ms,...,building_count,building_area_m2,road_count,road_length_m,poi_count,zone_area_km2,building_footprint_pct,building_count_per_km2,poi_count_per_km2,road_density_km_per_km2
0,87438411effffff,2024Q3,1,1,1,1,42.041,17.201,23.0,214.0,...,0.0,0.0,0.0,0.000000,0.0,4.448750,0.0,0.0,0.0,0.000000
1,87438411effffff,2024Q4,1,1,1,1,3.956,11.607,21.0,1802.0,...,0.0,0.0,0.0,0.000000,0.0,4.448750,0.0,0.0,0.0,0.000000
2,874384508ffffff,2025Q2,1,1,1,1,31.127,3.563,25.0,281.0,...,0.0,0.0,3.0,559.025795,0.0,4.415992,0.0,0.0,0.0,0.126591
3,874384508ffffff,2025Q3,1,4,2,4,117.812,5.593,31.0,1978.0,...,0.0,0.0,3.0,559.025795,0.0,4.415992,0.0,0.0,0.0,0.126591
4,874384508ffffff,2025Q4,1,5,4,5,128.126,12.992,26.0,156.0,...,0.0,0.0,3.0,559.025795,0.0,4.415992,0.0,0.0,0.0,0.126591


## Coverage check -- how much of the measured universe actually has population / OSM data

In [4]:
unique_zones = zone_quarter.drop_duplicates("h3_cell")
n = len(unique_zones)

has_pop = (unique_zones["population"] > 0).sum()
has_osm = (unique_zones[density_cols].sum(axis=1) > 0).sum()
has_both = ((unique_zones["population"] > 0) & (unique_zones[density_cols].sum(axis=1) > 0)).sum()
has_neither = ((unique_zones["population"] == 0) & (unique_zones[density_cols].sum(axis=1) == 0)).sum()

print(f"Measured zones (unique h3 cells): {n}")
print(f"  with population > 0:  {has_pop} ({has_pop/n:.1%})")
print(f"  with OSM features:    {has_osm} ({has_osm/n:.1%})")
print(f"  with both:            {has_both} ({has_both/n:.1%})")
print(f"  with neither (0/0):   {has_neither} ({has_neither/n:.1%}) -- e.g. desert/water-edge tiles")

Measured zones (unique h3 cells): 3400
  with population > 0:  3178 (93.5%)
  with OSM features:    3146 (92.5%)
  with both:            3036 (89.3%)
  with neither (0/0):   112 (3.3%) -- e.g. desert/water-edge tiles


## Save

`zone_quarter_table.parquet` is now the single input everything downstream reads: Experience
Index and Confidence Score (Phase 2), peer-group classification, anomaly detection, trend and
priority (Phase 3).

In [5]:
out_path = Path("../data/processed/zone_quarter_table.parquet")
zone_quarter.to_parquet(out_path, index=False)
print(f"Saved: {out_path} ({out_path.stat().st_size / 1024:.1f} KB)")
print("Columns:", list(zone_quarter.columns))

Saved: ..\data\processed\zone_quarter_table.parquet (676.6 KB)
Columns: ['h3_cell', 'quarter', 'n_tiles', 'tests', 'devices', 'tests_loaded_lat', 'download_mbps', 'upload_mbps', 'latency_ms', 'latency_loaded_ms', 'population', 'building_count', 'building_area_m2', 'road_count', 'road_length_m', 'poi_count', 'zone_area_km2', 'building_footprint_pct', 'building_count_per_km2', 'poi_count_per_km2', 'road_density_km_per_km2']


## Summary

- `data/processed/zone_quarter_table.parquet` -- one row per `(H3 res-7 cell, quarter)`, 13,597
  rows, joining Ookla measurements (per-quarter) with population and OSM density (static,
  broadcast across quarters). No rows added or dropped by the join; Ookla test counts unchanged.
- Zones with no population/OSM record are real zero-population, zero-feature zones (not missing
  data) -- filled with 0, except `zone_area_km2`, which is a geometric constant recomputed for
  every zone from `h3.cell_area` rather than inherited from the OSM join.

**Next:** build the peer-group composite classifier (commercial/urban-core, low-density
residential, industrial, rural/edge) on top of this table's density columns, per the brief's
explicit instruction not to classify by OSM land-use tag.